# Stage-1 Detection Results — Real-Time Evaluation\n\nThis notebook loads the actual EXP-10 YOLO11m best.pt checkpoint and runs real inference on both the **Validation** and **Test** sets.\nAll metrics are computed in real-time — nothing is hardcoded.\n\n**Model:** EXP-10 YOLO11m (640x640, AdamW, lr=0.001)\n**Checkpoint:** `runs/detect/EXP-10-YOLOv12m-640-200/weights/best.pt`

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\n!pip install ultralytics -q

In [ ]:
import os\nimport torch\nfrom ultralytics import YOLO\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nsns.set_theme(style='whitegrid')\nplt.rcParams['figure.figsize'] = (10, 6)\nplt.rcParams['figure.dpi'] = 120\n\nPROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'\nos.chdir(PROJECT_ROOT)\nprint(f'Working directory: {os.getcwd()}')\nprint(f'CUDA: {torch.cuda.is_available()}')\nif torch.cuda.is_available():\n    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load EXP-10 Best Checkpoint

In [ ]:
# Load the actual trained YOLO11m checkpoint\nYOLO_BEST_PT = 'runs/detect/EXP-10-YOLOv12m-640-200/weights/best.pt'\n\n# Handle the duplicate runs/detect path issue from original training\nif not os.path.exists(YOLO_BEST_PT):\n    YOLO_BEST_PT = 'runs/detect/runs/detect/EXP-10-YOLOv12m-640-200/weights/best.pt'\n\nprint(f'Loading checkpoint from: {YOLO_BEST_PT}')\nassert os.path.exists(YOLO_BEST_PT), f'Checkpoint NOT found at {YOLO_BEST_PT}!'\n\nmodel = YOLO(YOLO_BEST_PT)\nprint(f'\\nModel loaded successfully.')\nprint(f'Model type: {model.model.__class__.__name__}')\nprint(f'Number of classes: {model.model.nc}')

## 2. Evaluate on VALIDATION Set\nRunning real inference on the validation split to compute actual metrics.

In [ ]:
# Run validation on the VALIDATION set\nprint('='*60)\nprint('EVALUATING ON VALIDATION SET')\nprint('='*60)\n\nval_results = model.val(\n    data='dataset_yolo_single_class/data.yaml',\n    split='val',\n    imgsz=640,\n    conf=0.001,  # Low conf for mAP calculation\n    iou=0.5,\n    verbose=True\n)\n\nval_precision = val_results.box.mp * 100\nval_recall = val_results.box.mr * 100\nval_map50 = val_results.box.map50 * 100\nval_map5095 = val_results.box.map * 100\nval_f1 = (2 * val_precision * val_recall) / (val_precision + val_recall) if (val_precision + val_recall) > 0 else 0\n\nprint(f'\\n{"="*60}')\nprint(f'VALIDATION SET RESULTS')\nprint(f'{"="*60}')\nprint(f'Precision:  {val_precision:.2f}%')\nprint(f'Recall:     {val_recall:.2f}%')\nprint(f'F1-Score:   {val_f1:.2f}%')\nprint(f'mAP@50:     {val_map50:.2f}%')\nprint(f'mAP@50-95:  {val_map5095:.2f}%')

## 3. Evaluate on TEST Set\nRunning real inference on the completely blind test split.

In [ ]:
# Run validation on the TEST set\nprint('='*60)\nprint('EVALUATING ON TEST SET')\nprint('='*60)\n\ntest_results = model.val(\n    data='dataset_yolo_single_class/data.yaml',\n    split='test',\n    imgsz=640,\n    conf=0.001,\n    iou=0.5,\n    verbose=True\n)\n\ntest_precision = test_results.box.mp * 100\ntest_recall = test_results.box.mr * 100\ntest_map50 = test_results.box.map50 * 100\ntest_map5095 = test_results.box.map * 100\ntest_f1 = (2 * test_precision * test_recall) / (test_precision + test_recall) if (test_precision + test_recall) > 0 else 0\n\nprint(f'\\n{"="*60}')\nprint(f'TEST SET RESULTS')\nprint(f'{"="*60}')\nprint(f'Precision:  {test_precision:.2f}%')\nprint(f'Recall:     {test_recall:.2f}%')\nprint(f'F1-Score:   {test_f1:.2f}%')\nprint(f'mAP@50:     {test_map50:.2f}%')\nprint(f'mAP@50-95:  {test_map5095:.2f}%')

## 4. Validation vs Test Comparison Chart

In [ ]:
# Side-by-side comparison\nmetrics = ['Precision', 'Recall', 'F1-Score', 'mAP@50', 'mAP@50-95']\nval_vals = [val_precision, val_recall, val_f1, val_map50, val_map5095]\ntest_vals = [test_precision, test_recall, test_f1, test_map50, test_map5095]\n\ncomparison_df = pd.DataFrame({\n    'Metric': metrics,\n    'Validation Set': [f'{v:.2f}%' for v in val_vals],\n    'Test Set': [f'{v:.2f}%' for v in test_vals]\n})\ndisplay(comparison_df)\n\n# Bar chart\nimport numpy as np\nx = np.arange(len(metrics))\nwidth = 0.35\n\nfig, ax = plt.subplots(figsize=(12, 6))\nbars1 = ax.bar(x - width/2, val_vals, width, label='Validation Set', color='#2196F3')\nbars2 = ax.bar(x + width/2, test_vals, width, label='Test Set', color='#FF5722')\n\nax.set_ylabel('Percentage (%)', fontsize=12)\nax.set_title('EXP-10 YOLO11m Detection: Validation vs Test Set Metrics', fontsize=14, pad=15)\nax.set_xticks(x)\nax.set_xticklabels(metrics, fontsize=11)\nax.legend(fontsize=11)\nax.set_ylim(0, 100)\n\nfor bar in bars1 + bars2:\n    height = bar.get_height()\n    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),\n                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)\n\nplt.tight_layout()\nplt.savefig('detection_val_vs_test.png', dpi=150, bbox_inches='tight')\nplt.show()\nprint('\\nSource: Real-time inference using EXP-10 best.pt checkpoint.')

## 5. Summary\n\n**EXP-10 YOLO11m** is the best-performing detector among all evaluated configurations.\nThe validation and test metrics above are computed by loading the actual saved `best.pt` checkpoint and running real inference — no hardcoded values.